# Capstone — Content Review Prioritization

**Author:** Yusuf Ayman

**Lane:** Refresh / Content Opportunity Scoring

This notebook is the reproducible analysis behind the deployed research page.

## 1. Question

**Research question:** Which content pages should be reviewed first for a potential refresh or other improvement?

**Decision:** prioritize limited human review capacity. The output is decision-support, not an automatic editor.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

SEED=42
paths=[Path('data/raw/content_refresh_anonymized.csv'),Path('../../data/raw/content_refresh_anonymized.csv')]
data_path=next((p for p in paths if p.exists()),None)
if data_path is None: raise FileNotFoundError('Starter dataset not found.')
df=pd.read_csv(data_path)
TARGET='is_declining_label'
if TARGET not in df.columns: df[TARGET]=(df['trend_direction'].astype(str).str.lower()=='down').astype(int)
print('Dataset:',df.shape,'| Positive rate:',round(df[TARGET].mean(),4))

## 2. Data

The starter release is a 30,000-row × 44-column anonymized content snapshot. For the capstone model, `trend_direction`/`trend_pct` are evaluation-only because the decline proxy is derived from them. IDs are grouping keys only. Future information and baseline-generated fields are excluded.

In [ ]:
FORBIDDEN={TARGET,'trend_direction','trend_pct','content_id','client_id','score','reason_code','action_label','freshness_bucket','volume_bucket'}
features=[c for c in df.columns if c not in FORBIDDEN]
X=df[features].copy(); y=df[TARGET].astype(int); groups=df['client_id'].astype(str)
num=X.select_dtypes(include=np.number).columns.tolist(); cat=[c for c in features if c not in num]
print('Features used:',len(features),'| numeric:',len(num),'| categorical:',len(cat))

## 3. Methodology

The learned model is Logistic Regression with median imputation + missingness indicators for numeric features and most-frequent imputation + one-hot encoding for categorical features. The label is the observed starter trend proxy. Validation uses a client-grouped 80/20 split with seed 42.

In [ ]:
prep=ColumnTransformer([
 ('num',Pipeline([('imp',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),num),
 ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)
])
model=Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED))])
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
tr,te=next(gss.split(X,y,groups=groups))
assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te]))
model.fit(X.iloc[tr],y.iloc[tr])
prob=model.predict_proba(X.iloc[te])[:,1]; pred=(prob>=.50).astype(int); yt=y.iloc[te].to_numpy()
metrics={'split':'client_grouped_80_20','random_state':SEED,'train_rows':int(len(tr)),'test_rows':int(len(te)),'recall':float(recall_score(yt,pred,zero_division=0)),'precision':float(precision_score(yt,pred,zero_division=0)),'f1':float(f1_score(yt,pred,zero_division=0)),'roc_auc':float(roc_auc_score(yt,prob)),'base_rate':float(y.mean())}
print(json.dumps(metrics,indent=2))

## 4. Results (model vs frozen baseline on the same grouped holdout)

The baseline uses only the two pre-decision rule inputs: age and 90-day impressions. Both methods are evaluated on exactly the same test rows.

In [ ]:
def p_at_k(labels,scores,k):
 order=np.argsort(-np.asarray(scores))[:min(k,len(labels))]
 return float(np.asarray(labels)[order].mean())

test=df.iloc[te].copy()
base_flag=test['days_since_last_update'].fillna(0).ge(180)&test['impressions_90d'].fillna(0).ge(3000)
base_score=np.where(base_flag,test['impressions_90d'].fillna(0),0.0)
base_pred=base_flag.astype(int).to_numpy()
comparison=pd.DataFrame([
 {'method':'Frozen baseline','Recall':recall_score(yt,base_pred,zero_division=0),'Precision':precision_score(yt,base_pred,zero_division=0),'F1':f1_score(yt,base_pred,zero_division=0),'ROC-AUC':roc_auc_score(yt,base_score),'Precision@20':p_at_k(yt,base_score,20),'Precision@50':p_at_k(yt,base_score,50)},
 {'method':'Logistic Regression','Recall':metrics['recall'],'Precision':metrics['precision'],'F1':metrics['f1'],'ROC-AUC':metrics['roc_auc'],'Precision@20':p_at_k(yt,prob,20),'Precision@50':p_at_k(yt,prob,50)}
])
print(comparison.round(4).to_string(index=False))
print('Confusion matrix [TN FP; FN TP]:\n',confusion_matrix(yt,pred))

## 5. Limitations

The starter decline label is a proxy derived from `trend_direction`; it is not a causal or guaranteed future business outcome. One grouped split tests transfer across clients but does not prove universal future performance. The model identifies associations and ranking utility; it does not prove that refreshing a page causes recovery.

## 6. Ranked recommendations

1. **Refresh review:** stale + visible pages are the clearest first-pass candidates.
2. **Content diagnostic:** fresh pages with high model scores deserve investigation rather than automatic rewriting.
3. **Diagnostic review:** stale + low-visibility pages should be checked for opportunity before spending refresh effort.
4. **Monitor:** low-priority pages remain in the queue but do not consume immediate review capacity.

Every action remains subject to human review.

In [ ]:
# Build a public-safe ranked queue from the full starter snapshot.
full_prob=model.predict_proba(X)[:,1]
queue=df[['content_id','client_id','days_since_last_update','impressions_90d']].copy()
queue['review_score']=full_prob
queue['reason_code']=np.select([
 (queue.days_since_last_update.fillna(0)>=180)&(queue.impressions_90d.fillna(0)>=3000),
 (queue.days_since_last_update.fillna(0)<180)&(queue.review_score>=.5),
 (queue.days_since_last_update.fillna(0)>=180)&(queue.impressions_90d.fillna(0)<3000)],
 ['stale_and_visible','fresh_but_high_risk','stale_but_low_visibility'],default='recent_monitor')
queue['recommended_action']=queue['reason_code'].map({'stale_and_visible':'refresh_review','fresh_but_high_risk':'content_diagnostic','stale_but_low_visibility':'diagnostic_review','recent_monitor':'monitor'})
queue=queue.sort_values(['review_score','impressions_90d'],ascending=[False,False]).reset_index(drop=True)
queue['priority_rank']=np.arange(1,len(queue)+1)
out=Path('work/outputs/capstone_ranked_queue.csv'); out.parent.mkdir(parents=True,exist_ok=True); queue.to_csv(out,index=False)
metrics_path=Path('work/outputs/capstone_metrics.json'); metrics_path.write_text(json.dumps(metrics,indent=2))
print('Top 10 ranked candidates (pseudonymous IDs only):'); print(queue.head(10).to_string(index=False)); print('Saved:',out); print('Saved:',metrics_path)

## 7. Artifacts the paper embeds

The capstone produces a reproducible metrics receipt and ranked queue. The deployed paper in `docs/index.html` presents the public-safe methodology, measured grouped-holdout results, limitations, and action logic.

**Reproducibility:** seed 42; install `pandas numpy scikit-learn matplotlib`; run this notebook top-to-bottom from a fresh clone.

## ML-12 — 5-minute demo outline

- **0:00–0:45:** Question — which pages should receive human review first?
- **0:45–1:30:** Data — 30,000 × 44 starter snapshot; public-safe anonymized IDs.
- **1:30–2:30:** Method — Logistic Regression, leakage exclusions, client-grouped 80/20 split.
- **2:30–3:30:** Results — show the model-vs-baseline table and explain Recall/Precision.
- **3:30–4:30:** Error analysis — discuss false positives/negatives and why ranking is not certainty.
- **4:30–5:00:** Recommendation — use the queue to prioritize human investigation, never auto-publish/delete/redirect/rewrite.

### Social-post cut

I built a content-review prioritization workflow on the FlyRank internship dataset, using Logistic Regression with client-grouped validation and explicit leakage controls. The evaluated model provides measurable out-of-sample ranking/classification performance and turns the result into a human-reviewed action queue. The important part: the score is decision-support, not a claim that a refresh will cause recovery.

### Employer-facing summary

I built an end-to-end ML workflow for prioritizing content review, from problem framing and leakage-safe feature construction to a transparent baseline, grouped validation, error analysis, and an actionable ranked queue. It uses an anonymized 30,000-row content dataset and a client-grouped holdout to test whether the learned signal transfers across clients. The result is a reproducible decision-support artifact with explicit limitations rather than an opaque model demo.

## Self-check

- [x] Question, decision, data, methodology, results, limitations, recommendations, and artifacts are present.
- [x] Model and baseline use the same grouped holdout.
- [x] Leakage fields and IDs are excluded from features.
- [x] Metrics are computed from the current run rather than invented.
- [x] Ranked output and metrics receipt are generated.
- [x] ML-12 demo, social cut, and employer summary are included.
- [x] No causal claims or client-identifying details are included.